In [1]:
# 0 Lib import and decleration

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation

print("Setting up animated bar chart race...\n")

TEAL, PURPLE, NAVY = "#00C4A0", "#7B5EA7", "#0A0A2E"
RED, ORANGE, BLUE = "#E74C3C", "#F5A623", "#4EA8DE"
GOLD, GREY = "#C9A84C", "#95A5A6"

plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white"})

Setting up animated bar chart race...



In [2]:
# STEP 1: EV sales dataset
print("Step 1: Creating EV sales dataset...")
years = list(range(2010, 2024))

data_dict = {
    "Tesla":      [0.2, 0.5, 1.2, 2.6, 5.1, 12.5, 24.5, 36.7, 49.0, 66.0, 81.5, 77.2, 86.4, 95.0],
    "BYD":        [0.1, 0.3, 0.8, 1.5, 2.9, 6.0, 10.2, 19.3, 32.8, 59.3, 74.6, 101.2, 120.5, 130.0],
    "Volkswagen": [0.0, 0.2, 0.6, 1.8, 3.9, 8.2, 15.6, 22.1, 27.4, 31.2, 35.8, 41.0, 48.2, 52.0],
    "Li Auto":    [0.0, 0.0, 0.0, 0.0, 0.2, 1.4, 5.8, 13.8, 29.5, 38.2, 42.6, 48.0, 52.3, 55.0],
    "BMW":        [0.0, 0.1, 0.4, 1.1, 2.3, 5.0, 9.8, 14.2, 18.5, 21.0, 24.7, 27.5, 30.2, 32.0],
    "Geely":      [0.0, 0.0, 0.0, 0.3, 0.7, 1.6, 3.5, 7.2, 13.4, 18.6, 21.8, 25.0, 28.5, 31.0],
    "Hyundai":    [0.0, 0.0, 0.0, 0.0, 0.1, 0.6, 2.8, 7.4, 14.0, 22.5, 29.1, 33.5, 38.9, 42.0],
    "Nissan":     [0.0, 0.0, 0.1, 0.8, 2.4, 5.1, 8.9, 12.3, 14.1, 15.2, 14.8, 13.5, 12.1, 11.0],
}

df = pd.DataFrame(data_dict, index=years).T
print(f" Dataset: {df.shape[0]} makers × {df.shape[1]} years")

Step 1: Creating EV sales dataset...
 Dataset: 8 makers × 14 years


In [3]:
# STEP 2: Create frames
print("\n Step 2: Creating animation frames...")

def create_frames(df, frames_per_year=8):
    all_frames = []
    for i in range(len(df.columns) - 1):
        year_a, year_b = df.columns[i], df.columns[i + 1]
        data_a, data_b = df[year_a].values, df[year_b].values
        makers = df.index.tolist()
        
        for frame_num in range(frames_per_year):
            progress = frame_num / frames_per_year
            interp = data_a * (1 - progress) + data_b * progress
            frame_df = pd.DataFrame({"maker": makers, "value": interp})
            frame_df["year"] = year_a + progress
            frame_df = frame_df.sort_values("value", ascending=False)
            all_frames.append(frame_df)
    
    final = pd.DataFrame({"maker": makers, "value": df[df.columns[-1]].values, "year": df.columns[-1]})
    final = final.sort_values("value", ascending=False)
    all_frames.append(final)
    return all_frames

frames = create_frames(df, frames_per_year=8)
total_frames = len(frames)
print(f"Created {total_frames} frames")


 Step 2: Creating animation frames...
Created 105 frames


In [4]:
# STEP 3: Animate to GIF
print("\n🎬 Step 3: Rendering animation to GIF...")

maker_colours = {
    "Tesla": TEAL, "BYD": PURPLE, "Volkswagen": ORANGE, "Li Auto": BLUE,
    "BMW": GOLD, "Geely": RED, "Hyundai": GREY, "Nissan": "#1ABC9C",
}

fig, ax = plt.subplots(figsize=(12, 6))

def animate(frame_num):
    ax.clear()
    frame_data = frames[frame_num]
    year = int(frame_data["year"].iloc[0])
    top_data = frame_data.head(6)
    
    makers = top_data["maker"].tolist()
    values = top_data["value"].values
    colours = [maker_colours.get(m, GREY) for m in makers]
    
    ax.barh(range(len(makers)), values, color=colours, zorder=2)
    ax.set_yticks(range(len(makers)))
    ax.set_yticklabels(makers, fontsize=11, fontweight="bold")
    
    for i, (maker, val) in enumerate(zip(makers, values)):
        ax.text(val + 1.5, i, f"{val:.1f}M", va="center", fontsize=10, color=NAVY, fontweight="bold")
    
    ax.set_xlabel("EV Sales (Millions)", fontsize=11, fontweight="bold")
    ax.set_title(f"Global EV Sales Race\n{year}", fontsize=16, fontweight="bold", color=NAVY, pad=16)
    ax.set_xlim(0, 140)
    ax.grid(axis="x", alpha=0.3, zorder=1)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.invert_yaxis()

anim = animation.FuncAnimation(fig, animate, frames=total_frames, interval=50, repeat=True, blit=False)
anim.save("ev_sales_race.gif", writer="pillow", fps=20, dpi=100)
plt.close()
print("GIF saved successfully")


🎬 Step 3: Rendering animation to GIF...
GIF saved successfully


In [6]:
# STEP 4: Before/After
print("\nStep 4: Creating comparison...")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, year in zip(axes, [2010, 2023]):
    data = df[year].sort_values(ascending=False).head(6)
    colours = [maker_colours.get(m, GREY) for m in data.index]
    ax.barh(range(len(data)), data.values, color=colours, zorder=2)
    ax.set_yticks(range(len(data)))
    ax.set_yticklabels(data.index, fontsize=11, fontweight="bold")
    for i, val in enumerate(data.values):
        ax.text(val + 2, i, f"{val:.0f}M", va="center", fontsize=10, color=NAVY, fontweight="bold")
    ax.set_title(f"{year}", fontsize=14, fontweight="bold", color=NAVY)
    ax.set_xlim(0, 140)
    ax.grid(axis="x", alpha=0.2, zorder=1)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.invert_yaxis()

fig.suptitle("13 Years: Tesla to BYD Supremacy", fontsize=16, fontweight="bold", color=NAVY, y=1.02)
plt.tight_layout()
plt.savefig("v4_before_after.png", dpi=150, bbox_inches="tight")
plt.close()
print("Comparison saved")

print(f"\nVideo 4 Python complete — {total_frames} frames @ 20fps = {total_frames/20:.1f}s animation")


Step 4: Creating comparison...
Comparison saved

Video 4 Python complete — 105 frames @ 20fps = 5.2s animation
